In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ExpSineSquared, WhiteKernel
from gpr_predictions import Halogenation
import warnings

warnings.filterwarnings("ignore") # Ignore convergence warnings when running the GPR models

Data imports

In [11]:
data_yield = pd.read_csv('hte_halogenation_yields_for_gpr.csv')
data_conv = pd.read_csv('hte_halogenation_conversions_for_gpr.csv')

# Seperate the yield data into the different halogenations
chloro_yield = data_yield[data_yield['product'].str.contains('chloro')]
bromo_yield = data_yield[data_yield['product'].str.contains('bromo')]
iodo_yield = data_yield[data_yield['product'].str.contains('iodo')]

# Seperate the conversion data into the different halogenations
chloro_conv = data_conv[data_conv['product'].str.contains('chloro')]
bromo_conv = data_conv[data_conv['product'].str.contains('bromo')]
iodo_conv = data_conv[data_conv['product'].str.contains('iodo')]

# GPR Predictions for Stage 2 halogenation conditions prediction

Model parameters to screen

In [12]:
screening_bounds = [(1e-1, 1e1), (1e0, 1e2)]

# Kernels for screening
def get_kernels(ls_bounds):
    return {'RBF': 1.0 * RBF(length_scale=1e0, length_scale_bounds=ls_bounds)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          'Matern (nu = 3/2)': 1.0 * Matern(length_scale=1e0, length_scale_bounds=ls_bounds, nu=3/2)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          'Matern (nu = 5/2)': 1.0 * Matern(length_scale=1e0, length_scale_bounds=ls_bounds, nu=5/2)+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          #'Rational quadratic (RQ)': 1.0 * RationalQuadratic(length_scale=1e0, length_scale_bounds=ls_bounds),+ WhiteKernel(noise_level=1e0, noise_level_bounds=ls_bounds),
          #UNCOMMENT THE ABOVE LINES TO USE OTHER KERNELS. Tended to result in overfitting for many of the datasets used in this study
          }

Data storage

In [ ]:
# Initialise dataframe to store predictions
optimum_condition_df = pd.DataFrame(columns=['pred_opt_tfa', 
                            'pred_opt_yield',
                            'pred_opt_conv',
                            'kernel_yield',
                            'kernel_yield_error',
                            'kernel_conv',
                            'kernel_conv_error',
                            'length_scale_bounds',
                           ])

# Initialise lists to store GPR predictions
gpr_predictions_yield = []
gpr_predictions_conv = []